# Raster PCA maps (Radiometrics + DEM)

Run the cells in order to export **PC1, PC2 and PC3** as separate GeoTIFFs and preview them below. Install the dependencies in your notebook kernel if needed with `%pip install numpy rasterio scikit-learn matplotlib`.

Please Edit `DATA_DIR` and `INPUTS` to use another dataset. Relative paths start in the kernel’s working directory. The first raster sets the output grid. Each band contributes one variable, so use numeric geophysical grids. The Red Dome example corrects the Pot and Tho CRS tags to MGA zone 55. Use `None` when a raster’s CRS tag is correct.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.crs import CRS #this for Coordinates Reference system since the project uses GeoTIFFs
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent
from rasterio.warp import reproject
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler #It's possible also to test differnt scalers including robust scaler to remove outliers, MinMax is not advised

dir = Path(".")
maps = Path("Red_Dome_geophys_complete/mg001262")
data_feed = [
    ("Red Dome Area DEM.tif", None),
    (maps / "Pot_1262.tif", "EPSG:28355"),
    (maps / "Tho_1262.tif", "EPSG:28355"),
    (maps / "Ura_1262.tif", None),
]
output_dir = Path("pca_output")

In [ ]:
def read_bands(path, crs_override, grid):
    """Yield bands on the reference grid, with NaN (null pixels) outside valid coverage."""
    shape, transform, crs = grid
    with rasterio.open(path) as src:
        src_crs = CRS.from_user_input(crs_override) if crs_override else src.crs
        same_grid = (src.shape, src.transform, src_crs) == grid
        for band in src.read(masked=True, out_dtype="float64"):
            values = band.filled(np.nan)
            if same_grid:
                yield values
                continue
            if src_crs is None or crs is None:
                raise ValueError(f"A CRS is required to align {path}.") #this if statement is necessary to get the images of the same area to align in the same coordinate system

            # Pad the values and the validity mask to check interpolation at the edges.
            valid = np.isfinite(values)
            warp = dict(
                src_transform=src.transform * rasterio.Affine.translation(-2, -2),
                src_crs=src_crs, dst_transform=transform, dst_crs=crs,
                resampling=Resampling.bilinear, dst_nodata=0,
            )
            aligned = np.zeros(shape, dtype="float64")
            coverage = np.zeros_like(aligned)
            reproject(np.pad(np.where(valid, values, 0), 2), aligned, **warp)
            reproject(np.pad(valid.astype("float64"), 2), coverage, **warp)
            yield np.where(np.isclose(coverage, 1, rtol=0, atol=1e-12), aligned, np.nan)

In [ ]:
paths = [(dir / path, override) for path, override in maps]
if not paths:
    raise ValueError("Set at least one input raster.")

first_path, first_override = paths[0]
with rasterio.open(first_path) as src:
    crs = CRS.from_user_input(first_override) if first_override else src.crs
    grid = src.shape, src.transform, crs

bands = np.stack([band for path, override in paths for band in read_bands(path, override, grid)])
valid = np.isfinite(bands).all(axis=0)
if len(bands) < 3 or valid.sum() < 3:
    raise ValueError("PCA needs at least three bands and three common valid pixels.")

pixels = StandardScaler().fit_transform(bands[:, valid].T)
if not np.any(pixels):
    raise ValueError("All input bands are constant in the common valid area.")
pca = PCA(n_components=3, svd_solver="full") #in this project 3 PC will be outputted
scores = pca.fit_transform(pixels)
print(f"Common valid pixels: {valid.sum():,}")

outputs = [ / f"PC{i}.tif" for i in range(1, 4)]
if {out.resolve() for out in outputs} & {path.resolve() for path, _ in paths}:
    raise ValueError("Choose an output directory that will not overwrite an input raster.")
output_dir.mkdir(parents=True, exist_ok=True)

(height, width), transform, crs = grid
profile = dict(
    driver="GTiff", height=height, width=width, count=1, dtype="float32",
    transform=transform, crs=crs, nodata=np.nan,
)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, path, score, variance in zip(axes, outputs, scores.T, pca.explained_variance_ratio_):
    image = np.full((height, width), np.nan, dtype="float32")
    image[valid] = score
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(image, 1)
        dst.set_band_description(1, path.stem)
        dst.update_tags(1, EXPLAINED_VARIANCE_PERCENT=f"{variance * 100:.8f}")

    limit = np.percentile(np.abs(score), 98) or 1
    view = ax.imshow(image, extent=plotting_extent(image, transform), cmap="RdBu_r",
                     vmin=-limit, vmax=limit, interpolation="nearest")
    ax.set_title(f"{path.stem} | {variance:.2%} of variance")
    ax.ticklabel_format(style="plain", useOffset=False)
    fig.colorbar(view, ax=ax, shrink=0.8, label="PC score")
plt.show()